# Analyse af udgiftskrav

Denne notesbog demonstrerer, hvordan man opretter agenter, der bruger plugins til at behandle rejseudgifter fra lokale kvitteringsbilleder, generere en e-mail til udgiftskrav og visualisere udgiftsdata ved hjælp af et cirkeldiagram. Agenter vælger dynamisk funktioner baseret på opgavens kontekst.

Trin:
1. OCR-agent behandler det lokale kvitteringsbillede og udtrækker rejseudgiftsdata.
2. Email-agent genererer en e-mail til udgiftskravet.

### Eksempel på en rejseudgiftssituation:
Forestil dig, at du er en medarbejder, der rejser for et forretningsmøde i en anden by. Din virksomhed har en politik om at refundere alle rimelige rejserelaterede udgifter. Her er en oversigt over potentielle rejseudgifter:
- Transport:
Flybillet for en returrejse fra din hjemby til destinationen.
Taxi eller kørselstjenester til og fra lufthavnen.
Lokal transport i destinationsbyen (som offentlig transport, lejebiler eller taxaer).

- Indkvartering:
Hotelophold i tre nætter på et mellemklasse forretningshotel tæt på mødestedet.

- Måltider:
Dagligt måltidstillæg til morgenmad, frokost og aftensmad baseret på virksomhedens diætpulje.

- Diverse udgifter:
Parkeringsafgifter i lufthavnen.
Internettillæg på hotellet.
Drikkepenge eller små serviceafgifter.

- Dokumentation:
Du indsender alle kvitteringer (fly, taxa, hotel, måltider osv.) og en udfyldt udgiftsrapport til refusion.


## Importer nødvendige biblioteker

Importer de nødvendige biblioteker og moduler til notebooken.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definér Udgiftsmodeller

Opret en Pydantic-model for individuelle udgifter og en ExpenseFormatter-klasse til at konvertere en brugerforespørgsel til strukturerede udgiftsdata.

Hver udgift vil blive repræsenteret i formatet:
`{'date': '07-Mar-2025', 'description': 'fly til destination', 'amount': 675.99, 'category': 'Transport'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Generering af e-mail

Opret en værktøjsfunktion til at generere en e-mail til indsendelse af et udgiftskrav.
- Dette værktøj bruger `@tool` dekoratoren fra Microsoft Agent Framework.
- Den beregner det samlede beløb for udgifterne og formaterer detaljerne til en e-mailtekst.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Værktøj til Udtrækning af Rejseudgifter fra Kvitteringsbilleder

Opret en værktøjsfunktion til at udtrække rejseudgifter fra kvitteringsbilleder.
- Dette værktøj bruger `@tool` dekoratoren fra Microsoft Agent Framework.
- Det læser kvitteringsbilledet, koder det som base64 og returnerer data-URI'en til agenten for at analysere.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Behandling af udgifter

Definer agenterne og forbind dem i en sekventiel arbejdsgang ved hjælp af `WorkflowBuilder`.
- OCR-agenten udtrækker strukturerede udgiftsdata fra kvitteringsbilledet ved hjælp af værktøjet `load_receipt_image`.
- Email-agenten tager de udtrukne data og genererer en professionel udgiftsanmodningsmail ved hjælp af værktøjet `generate_expense_email`.
- `WorkflowBuilder` med `add_edge` opretter en sekventiel pipeline: OCR-agent → Email-agent.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hovedfunktion

Byg den sekventielle arbejdsgang og kør den for at behandle kvitteringsbilledet og generere e-mailen til udgiftsrefusion.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, bedes du være opmærksom på, at automatiske oversættelser kan indeholde fejl eller unøjagtigheder. Det oprindelige dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der måtte opstå som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
